In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
import sys
from pathlib import Path

# Load the observed data
df = pd.read_parquet("../temp_data/similarity_results_only_exact_match.parquet")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../temp_data/similarity_results_only_exact_match.parquet'

In [ ]:
# Check if there are I/L swap pairs (sequence1 vs sequence2)
print("Sample sequence pairs:")
for i in range(min(5, len(df))):
    print(f"  {df['sequence1'].iloc[i]} (z={df['charge1'].iloc[i]}) vs {df['sequence2'].iloc[i]} (z={df['charge2'].iloc[i]})")

# Check the wildcard column
print(f"\nWildcard values: {df['wildcard'].unique()}")
print(f"\nTotal comparisons: {len(df)}")

# Check charge state distribution
print(f"\nCharge1 values: {df['charge1'].unique()}")
print(f"Charge2 values: {df['charge2'].unique()}")

# Check how many have matching charge states
same_charge = df['charge1'] == df['charge2']
print(f"\nSame charge comparisons: {same_charge.sum()} / {len(df)}")
print(f"Different charge comparisons: {(~same_charge).sum()}")

In [ ]:
# Filter to only same-charge comparisons
df_same_charge = df[df['charge1'] == df['charge2']].copy()
print(f"Filtered to same-charge comparisons: {len(df_same_charge)} pairs")

# Summary statistics for key metrics (same charge only)
metrics = ['mse', 'ruzicka_similarity_1', 'ruzicka_similarity_2', 'spectral_angle', 'pearson_correlation', 'weighted_dot_product']
print("\nSummary statistics for observed data (same charge only):")
df_same_charge[metrics].describe()

In [ ]:
metrics_to_plot = ["mse", "spectral_angle"]

fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(12, 4))

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx]
    values = df_same_charge[metric].dropna()

    # Histogram
    ax.hist(values, bins=50, alpha=0.7, color='steelblue', edgecolor='black', linewidth=0.5)

    # Add mean and median lines
    mean_val = values.mean()
    median_val = values.median()
    ax.axvline(mean_val, color='red', linestyle='-', linewidth=2, label=f'Mean: {mean_val:.4f}')
    ax.axvline(median_val, color='orange', linestyle='--', linewidth=2, label=f'Median: {median_val:.4f}')

    ax.set_xlabel(f"{metric}")
    ax.set_ylabel("Frequency")
    ax.set_title(f"{metric} Distribution (Observed, Same Charge)")
    if metric == "ruzicka_similarity_2":
        ax.legend(loc="upper left")
    else:    
        ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

plt.suptitle(f"Observed I/L Sibling Spectra (n={len(df_same_charge)}, Same Charge)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

metrics_grid = [
    ("mse", "MSE", "upper right"),
    ("ruzicka_similarity_2", "Ruzicka (L2 norm)", "upper left"),
    ("weighted_dot_product", "Weighted Dot Product", "upper left"),
    ("pearson_correlation", "Pearson Correlation", "upper left"),
    ("spectral_angle", "Spectral Angle", "upper left"),
    # ("gnps_score", "GNPS Score", "upper left"),
]

for idx, (metric, label, legend_loc) in enumerate(metrics_grid):
    row, col = idx // 3, idx % 3
    ax = axes[row, col]
    values = df_same_charge[metric].dropna()

    # Histogram
    ax.hist(values, bins=50, alpha=0.7, color='steelblue', edgecolor='black', linewidth=0.5)

    # Add mean and median lines
    mean_val = values.mean()
    median_val = values.median()
    ax.axvline(mean_val, color='red', linestyle='-', linewidth=2, label=f'Mean: {mean_val:.4f}')
    ax.axvline(median_val, color='orange', linestyle='--', linewidth=2, label=f'Median: {median_val:.4f}')

    ax.set_xlabel(label)
    ax.set_ylabel("Frequency")
    ax.set_title(f"{label} Distribution")
    ax.legend(loc=legend_loc, fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle(f"Observed I/L Sibling Spectra (n={len(df_same_charge)}, Same Charge)", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("../temp_data/observed_il_metrics_figure.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# MSE
ax = axes[0]
values = df_same_charge["mse"].dropna()
ax.hist(values, bins=50, alpha=0.7, color='steelblue', edgecolor='black', linewidth=0.5)
mean_val, median_val = values.mean(), values.median()
ax.axvline(mean_val, color='red', linestyle='-', linewidth=2, label=f'Mean: {mean_val:.4f}')
ax.axvline(median_val, color='orange', linestyle='--', linewidth=2, label=f'Median: {median_val:.4f}')
ax.set_xlabel("MSE")
ax.set_ylabel("Frequency")
ax.set_title("MSE (Observed)")
ax.legend(loc='upper right', fontsize=8)
ax.grid(True, alpha=0.3)

# Spectral angle
ax = axes[1]
values = df_same_charge["spectral_angle"].dropna()
ax.hist(values, bins=50, alpha=0.7, color='steelblue', edgecolor='black', linewidth=0.5)
mean_val, median_val = values.mean(), values.median()
ax.axvline(mean_val, color='red', linestyle='-', linewidth=2, label=f'Mean: {mean_val:.4f}')
ax.axvline(median_val, color='orange', linestyle='--', linewidth=2, label=f'Median: {median_val:.4f}')
ax.set_xlabel("Spectral angle")
ax.set_ylabel("Frequency")
ax.set_title("Spectral angle (Observed)")
ax.legend(loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)

# Weighted Dot Product
ax = axes[2]
values = df_same_charge["weighted_dot_product"].dropna()
ax.hist(values, bins=50, alpha=0.7, color='steelblue', edgecolor='black', linewidth=0.5)
mean_val, median_val = values.mean(), values.median()
ax.axvline(mean_val, color='red', linestyle='-', linewidth=2, label=f'Mean: {mean_val:.4f}')
ax.axvline(median_val, color='orange', linestyle='--', linewidth=2, label=f'Median: {median_val:.4f}')
ax.set_xlabel("Weighted Dot Product")
ax.set_ylabel("Frequency")
ax.set_title("Weighted Dot Product (Observed)")
ax.legend(loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle(f"Observed I/L Sibling Spectra (n={len(df_same_charge)}, Same Charge)", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("../temp_data/observed_il_simplified_figure.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
project_root = Path.cwd().parent
sys.path.append(str(project_root))

from metrics.get_metrics import metrics_comparison

# Load the predicted intensity data
peptides_predictions = pd.read_csv("../temp_data/peptides_predictions.csv")
peptides_switch_predictions = pd.read_csv("../temp_data/peptides_switch_predictions.csv")

# Create ID column for predictions (needed for metrics_comparison)
peptides_predictions["seq_charge"] = (
    peptides_predictions["peptide_sequences"]
    + "/"
    + peptides_predictions["precursor_charges"].astype(str)
)
unique_combinations = peptides_predictions["seq_charge"].unique()
seq_charge_to_id = {combo: i for i, combo in enumerate(unique_combinations)}
peptides_predictions["ID"] = peptides_predictions["seq_charge"].map(seq_charge_to_id)

# For switched predictions, we need to map back to original sequences
# Load original peptides to create the mapping
from seq_utils.fasta_to_peptides import create_tryptic_peptides
from seq_utils.peptide import remove_non_il, remove_ux_containing, switch_random_il

fasta_file = "../fasta/UP000005640_9606.fasta"
peptides = create_tryptic_peptides(fasta_file)
peptides = remove_non_il(peptides)
peptides = remove_ux_containing(peptides)
peptides = np.array(peptides)

# Sample same peptides as figure1
rng = np.random.default_rng(42)
indices = rng.choice(len(peptides), size=20000, replace=False)
peptides = peptides[indices]
peptides_switched = np.array([switch_random_il(peptide) for peptide in peptides])

# Create mapping from switched to original
switched_to_orig = dict(zip(peptides_switched, peptides))

peptides_switch_predictions["seq_charge"] = (
    peptides_switch_predictions["peptide_sequences"]
    + "/"
    + peptides_switch_predictions["precursor_charges"].astype(str)
)
peptides_switch_predictions["orig_seq_charge"] = peptides_switch_predictions.apply(
    lambda row: switched_to_orig.get(row["peptide_sequences"], row["peptide_sequences"])
    + "/"
    + str(row["precursor_charges"]),
    axis=1,
)
peptides_switch_predictions["ID"] = peptides_switch_predictions["orig_seq_charge"].map(seq_charge_to_id)

# Compute similarity metrics for predicted data
score_df_predicted = metrics_comparison(peptides_predictions, peptides_switch_predictions)
print(f"Predicted data: {len(score_df_predicted)} comparisons")
print(f"Observed data (same charge): {len(df_same_charge)} comparisons")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

metrics_config = [
    ("mse", "MSE", "upper right"),
    ("spectral_angle", "Spectral angle", "upper left"),
    ("weighted_dot_product", "Weighted Dot Product", "upper left"),
]

for idx, (metric, label, legend_loc) in enumerate(metrics_config):
    ax = axes[idx]
    
    # Observed data (same charge) - histogram with counts
    obs_values = df_same_charge[metric].dropna()
    ax.hist(obs_values, bins=50, alpha=0.7, color='steelblue', edgecolor='black', 
            linewidth=0.5, label=f'Observed (n={len(obs_values)})')
    
    # Add mean/median for observed
    obs_mean = obs_values.mean()
    obs_median = obs_values.median()
    ax.axvline(obs_mean, color='darkblue', linestyle='-', linewidth=2, label=f'Obs Mean: {obs_mean:.4f}')
    ax.axvline(obs_median, color='blue', linestyle='--', linewidth=2, label=f'Obs Median: {obs_median:.4f}')
    
    # Predicted data - density on secondary y-axis (hidden)
    pred_values = score_df_predicted[metric].dropna()
    ax2 = ax.twinx()
    
    # KDE for predicted data
    pred_kde = stats.gaussian_kde(pred_values)
    pred_x = np.linspace(pred_values.min(), pred_values.max(), 200)
    pred_y = pred_kde(pred_x)
    ax2.fill_between(pred_x, pred_y, alpha=0.4, color='coral', label=f'Predicted (n={len(pred_values)})')
    ax2.plot(pred_x, pred_y, color='coral', linewidth=2)
    
    # Add mean/median for predicted
    pred_mean = pred_values.mean()
    pred_median = pred_values.median()
    ax.axvline(pred_mean, color='darkred', linestyle='-', linewidth=2, label=f'Pred Mean: {pred_mean:.4f}')
    ax.axvline(pred_median, color='red', linestyle='--', linewidth=2, label=f'Pred Median: {pred_median:.4f}')
    
    # Hide the secondary y-axis ticks and labels
    ax2.set_yticks([])
    ax2.set_ylabel('')
    # Force the secondary y-axis to start at 0
    ax2.set_ylim(bottom=0)
    
    ax.set_xlabel(label)
    ax.set_ylabel("Frequency (Observed)")
    ax.set_title(f"{label}")
    
    # Combine legends from both axes
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc=legend_loc, fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle("I/L Sibling Spectra: Observed (Histogram) vs ML-Predicted (Density)", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("../temp_data/observed_vs_predicted_overlay.png", dpi=300, bbox_inches='tight')
plt.show()